In [ ]:
!nvidia-smi #Make sure at least L4+ GPU (I dont think not compatable with H100)

In [ ]:
pip install torch==2.5.0 torchvision==0.20.0 torchaudio==2.5.0 --index-url https://download.pytorch.org/whl/cu124

In [ ]:
!pip install natten==0.17.5+torch250cu124 -f https://whl.natten.org

In [ ]:
!git clone --branch nathan https://github.com/bealowman/alzheimers-detection-working.git
%cd alzheimers-detection-working/MedViTV2-main


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
#I added the folder as a shortcut to main drive because pathing to shared drive is wonky so do that before running
import os
print(os.listdir('/content/drive/MyDrive/SP26_dementia_data/processed_data'))

In [ ]:
!pip install -r requirements.txt

In [ ]:
with open('main.py', 'r') as f:
    content = f.read()

old = "    parser.add_argument('--checkpoint_path', type=str, default='./checkpoint/MedViT_tiny.pth', help='Path to the checkpoint file.')"
new = old + "\n    parser.add_argument('--dataset_dir', type=str, default=None, help='Path to dataset root directory.')"

with open('main.py', 'w') as f:
    f.write(content.replace(old, new))

with open('main.py', 'r') as f:
    for line in f:
        if 'dataset_dir' in line:
            print("Patch:", line.strip())

Patch: parser.add_argument('--dataset_dir', type=str, default=None, help='Path to dataset root directory.')


In [ ]:
# with open('datasets.py', 'r') as f:
#     content = f.read()

# old = "    else:\n        raise NotImplementedError()"

# new = """    elif args.dataset == 'Dementia':
#         nb_classes = 4
#         drive_dir = getattr(args, 'dataset_dir', None)
#         downloader = DementiaDatasetDownloader(drive_dir=drive_dir)
#         train_dir, val_dir = downloader.get_dataset()
#         print(f"Dementia dataset is available at: train={train_dir} val={val_dir}")
#         train_dataset = datasets.ImageFolder(root=train_dir, transform=train_transform)
#         test_dataset = datasets.ImageFolder(root=val_dir, transform=test_transform)
#         return train_dataset, test_dataset, nb_classes
#     else:
#         raise NotImplementedError()"""

# with open('datasets.py', 'w') as f:
#     f.write(content.replace(old, new))

# with open('datasets.py', 'r') as f:
#     for line in f:
#         if 'Dementia' in line:
#             print("Found:", line.strip())

In [ ]:
!python main.py \
  --model_name 'MedViT_small' \
  --dataset 'Dementia' \
  --dataset_dir '/content/drive/MyDrive/SP26_dementia_data/processed_data' \
  --pretrained False \
  --save_dir '/content/drive/MyDrive/alzheimers_checkpoints'

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.12/dist-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
Using cuda:0 device.
Copying Dementia train split: /content/drive/MyDrive/SP26_dementia_data/processed_data/train -> data/Dementia-Dataset/train
Train copy complete.
Copying Dementia val split: /content/drive/MyDrive/SP26_dementia_data/processed_data/val -> data/Dementia-Dataset/val
Val copy complete.
Dementia dataset is available at: train=data/Dementia-Dataset/train val=data/Dementia-Dataset/val
initialize_weights...
Dataset ImageFolder
    Number of

In [ ]:
# [epoch 98] train_loss: 0.128 val_accuracy: 0.9939 precision: 0.9940 recall: 0.9939 specificity: 0.9980 f1_score: 0.9939 auc: 0.9999 overall_accuracy: 0.9939


In [ ]:
!python Heatmap/GradCam_MedViT_large.py --dataset 'Dementia' --image-path '/content/drive/MyDrive/alzheimers_checkpoints/confusion_matrix.png'

In [ ]:
!python evaluate.py \
  --checkpoint_path '/content/drive/MyDrive/alzheimers_checkpoints/MedViT_small_Dementia.pth' \
  --test_dir '/content/drive/MyDrive/SP26_dementia_data/processed_data/test' \
  --model_name 'MedViT_small' \
  --cm_save_path '/content/drive/MyDrive/alzheimers_checkpoints/confusion_matrix.png'